<a href="https://colab.research.google.com/github/engyousefk04-dot/Big-data-Assignment-3/blob/main/Airbnb_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Airbnb Price Prediction using Spark ML

This notebook implements only the required assignment steps:
1. Load the data
2. Explore the data
3. Split the data
4. Prepare features using VectorAssembler
5. Build and train Linear Regression model
6. Make predictions on training data
7. Test the model
8. Try adding more required features

## Start Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Airbnb Price Prediction").getOrCreate()

## Step 1: Load the Data

In [2]:
df = spark.read.csv(
    "airbnb_price_prediction_sample.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|   room_type|   neighborhood| price|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
|  1|      1.0|     1.0| 2.0|           3|             2|               57|                74.2|    Apartment| Shared room|Garden District|146.65|
|  2|      3.0|     1.0| 1.0|           2|             2|               59|                85.2|    Apartment| Shared room|         Marina| 252.6|
|  3|      3.0|     1.0| 1.0|           4|             3|              207|                96.1|        House| Shared room|    City Center|274.16|
|  4|      1.0|     1.0| 1.0|           2|             1|               97|                72.9| Private room| Shared 

## Step 2: Explore the Data

In [3]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- price: double (nullable = true)



In [4]:
df.select("bathrooms", "bedrooms", "price").show()

+---------+--------+------+
|bathrooms|bedrooms| price|
+---------+--------+------+
|      1.0|     1.0|146.65|
|      3.0|     1.0| 252.6|
|      3.0|     1.0|274.16|
|      1.0|     1.0|168.71|
|      4.0|     3.0|431.81|
|      1.0|     1.0|195.22|
|      2.5|     1.0|227.21|
|      1.5|     1.0|259.44|
|      1.0|     1.0|149.63|
|      4.0|     2.0|363.65|
|      3.0|     3.0|429.59|
|      1.0|     4.0|450.65|
|      1.0|     4.0|429.74|
|      4.0|     1.0|319.03|
|      2.0|     4.0|482.72|
|      2.5|     2.0|346.94|
|      1.0|     1.0|252.54|
|      1.0|     2.0|272.25|
|      3.0|     4.0|504.45|
|      1.0|     3.0|362.57|
+---------+--------+------+
only showing top 20 rows


## Step 3: Split the Data

In [5]:
trainDF, testDF = df.randomSplit([0.8, 0.2], seed=42)

print("Training Rows:", trainDF.count())
print("Testing Rows:", testDF.count())

Training Rows: 98
Testing Rows: 22


## Step 4: Prepare Features

In [6]:
from pyspark.ml.feature import VectorAssembler

vecAssembler = VectorAssembler(
    inputCols=["bathrooms", "bedrooms"],
    outputCol="features"
)

## Apply Transformation on Training Data

In [7]:
vecTrainDF = vecAssembler.transform(trainDF)

vecTrainDF.select("bathrooms", "bedrooms", "features", "price").show()

+---------+--------+---------+------+
|bathrooms|bedrooms| features| price|
+---------+--------+---------+------+
|      1.0|     1.0|[1.0,1.0]|146.65|
|      3.0|     1.0|[3.0,1.0]| 252.6|
|      1.0|     1.0|[1.0,1.0]|168.71|
|      4.0|     3.0|[4.0,3.0]|431.81|
|      1.0|     1.0|[1.0,1.0]|195.22|
|      1.5|     1.0|[1.5,1.0]|259.44|
|      4.0|     2.0|[4.0,2.0]|363.65|
|      3.0|     3.0|[3.0,3.0]|429.59|
|      1.0|     4.0|[1.0,4.0]|450.65|
|      1.0|     4.0|[1.0,4.0]|429.74|
|      2.0|     4.0|[2.0,4.0]|482.72|
|      2.5|     2.0|[2.5,2.0]|346.94|
|      1.0|     1.0|[1.0,1.0]|252.54|
|      1.0|     2.0|[1.0,2.0]|272.25|
|      3.0|     4.0|[3.0,4.0]|504.45|
|      2.5|     1.0|[2.5,1.0]|262.21|
|      1.5|     2.0|[1.5,2.0]|283.55|
|      1.0|     4.0|[1.0,4.0]|447.73|
|      1.0|     1.0|[1.0,1.0]|199.55|
|      3.0|     2.0|[3.0,2.0]|335.79|
+---------+--------+---------+------+
only showing top 20 rows


## Step 5: Build the Model

In [8]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction"
)

## Train the Model

In [9]:
lrModel = lr.fit(vecTrainDF)

## Step 6: Make Predictions on Training Data

In [10]:
predDF = lrModel.transform(vecTrainDF)

predDF.select(
    "bathrooms",
    "bedrooms",
    "features",
    "price",
    "prediction"
).show()

+---------+--------+---------+------+------------------+
|bathrooms|bedrooms| features| price|        prediction|
+---------+--------+---------+------+------------------+
|      1.0|     1.0|[1.0,1.0]|146.65|205.81981563880692|
|      3.0|     1.0|[3.0,1.0]| 252.6| 269.0154678055155|
|      1.0|     1.0|[1.0,1.0]|168.71|205.81981563880692|
|      4.0|     3.0|[4.0,3.0]|431.81| 459.4643705517651|
|      1.0|     1.0|[1.0,1.0]|195.22|205.81981563880692|
|      1.5|     1.0|[1.5,1.0]|259.44|221.61872868048408|
|      4.0|     2.0|[4.0,2.0]|363.65|380.03883222031743|
|      3.0|     3.0|[3.0,3.0]|429.59| 427.8665444684108|
|      1.0|     4.0|[1.0,4.0]|450.65| 444.0964306331499|
|      1.0|     4.0|[1.0,4.0]|429.74| 444.0964306331499|
|      2.0|     4.0|[2.0,4.0]|482.72| 475.6942567165041|
|      2.5|     2.0|[2.5,2.0]|346.94|  332.642093095286|
|      1.0|     1.0|[1.0,1.0]|252.54|205.81981563880692|
|      1.0|     2.0|[1.0,2.0]|272.25|285.24535397025454|
|      3.0|     4.0|[3.0,4.0]|5

## Step 7: Test the Model

In [11]:
vecTestDF = vecAssembler.transform(testDF)

predTestDF = lrModel.transform(vecTestDF)

predTestDF.select(
    "bathrooms",
    "bedrooms",
    "features",
    "price",
    "prediction"
).show()

+---------+--------+---------+------+------------------+
|bathrooms|bedrooms| features| price|        prediction|
+---------+--------+---------+------+------------------+
|      3.0|     1.0|[3.0,1.0]|274.16| 269.0154678055155|
|      2.5|     1.0|[2.5,1.0]|227.21|253.21655476383833|
|      1.0|     1.0|[1.0,1.0]|149.63|205.81981563880692|
|      4.0|     1.0|[4.0,1.0]|319.03|300.61329388886975|
|      1.0|     3.0|[1.0,3.0]|362.57| 364.6708923017022|
|      4.0|     1.0|[4.0,1.0]|255.15|300.61329388886975|
|      2.0|     1.0|[2.0,1.0]|217.86|237.41764172216122|
|      4.0|     3.0|[4.0,3.0]|445.83| 459.4643705517651|
|      3.0|     2.0|[3.0,2.0]|336.57|348.44100613696315|
|      1.0|     3.0|[1.0,3.0]|362.99| 364.6708923017022|
|      2.5|     2.0|[2.5,2.0]|302.99|  332.642093095286|
|      4.0|     3.0|[4.0,3.0]|452.07| 459.4643705517651|
|      4.0|     1.0|[4.0,1.0]|284.39|300.61329388886975|
|      3.0|     4.0|[3.0,4.0]|494.39|507.29208279985846|
|      1.0|     1.0|[1.0,1.0]| 

## Try: Improve the Model by Adding More Features

In [12]:
inputCols = [
    "bathrooms",
    "bedrooms",
    "beds",
    "accommodates",
    "review_scores_rating"
]

vecAssembler2 = VectorAssembler(
    inputCols=inputCols,
    outputCol="features"
)

vecTrainDF2 = vecAssembler2.transform(trainDF)
vecTestDF2 = vecAssembler2.transform(testDF)

lr2 = LinearRegression(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction"
)

lrModel2 = lr2.fit(vecTrainDF2)

predTestDF2 = lrModel2.transform(vecTestDF2)

predTestDF2.select(
    "bathrooms",
    "bedrooms",
    "beds",
    "accommodates",
    "review_scores_rating",
    "features",
    "price",
    "prediction"
).show()

+---------+--------+----+------------+--------------------+--------------------+------+------------------+
|bathrooms|bedrooms|beds|accommodates|review_scores_rating|            features| price|        prediction|
+---------+--------+----+------------+--------------------+--------------------+------+------------------+
|      3.0|     1.0| 1.0|           4|                96.1|[3.0,1.0,1.0,4.0,...|274.16|300.83402785585554|
|      2.5|     1.0| 1.0|           3|                78.0|[2.5,1.0,1.0,3.0,...|227.21|236.99455295707094|
|      1.0|     1.0| 1.0|           3|                72.0|[1.0,1.0,1.0,3.0,...|149.63|177.42904541647084|
|      4.0|     1.0| 2.0|           4|                85.0|[4.0,1.0,2.0,4.0,...|319.03|316.23405506984807|
|      1.0|     3.0| 4.0|           6|                70.2|[1.0,3.0,4.0,6.0,...|362.57|322.59991992498726|
|      4.0|     1.0| 1.0|           2|                77.5|[4.0,1.0,1.0,2.0,...|255.15| 270.3090427818572|
|      2.0|     1.0| 2.0|           4

## Stop Spark Session

In [13]:
spark.stop()